# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import ollama

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [3]:
# Initialize Ollama client
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
model = "gpt-oss:20b"  

In [4]:
import requests

# Cel servet ollama
try:
    requests.get("http://localhost:11434/v1/models", timeout=3)
    print("Ollama server is running.")
except requests.RequestException as e:
    print(f"Error occurred while checking Ollama server: {e}")

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

Ollama server is running.


In [5]:
response = ollama.chat.completions.create(model="gpt-oss:20b", 
                                          messages=[{'role': "system", "content": "You are a helpful assistant"},
                                                    {'role': "user", "content": "Hi! I'm Feb!"}])
print(response.choices[0].message.content)

Hello Feb! 👋 How can I assist you today?


In [6]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/
https://edwarddo

In [10]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [11]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio / resume',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'partner site',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {model}...")
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-oss:20b...
Found 10 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills showcase', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'press release',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'partner site',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b...
Found 19 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'company page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'about page', 'url': 'https://huggingface.co/huggingface/about'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'chat page', 'url': 'https://huggingface.co/chat'},
  {'type': 'support page', 'url': 'https://huggingface.co/support'},
  {'type': 'brand asset

## Second step: make the brochure!

Assemble all the details into another prompt to gpt-oss:20b

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b...
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Search models, datasets, users...
Ctrl K
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
thinkingmachines/Inkling-2-Mini
Updated
about 20 hours ago
•
41.9k
•
5.53k
unsloth/Inkling-2-Mini-GGUF
Updated
about 12 hours ago
•
1.29M
•
1.14k
inworld/voice-isolator
Updated
10 days ago
•
750
AtomicChat/Inkling-2-Mini-Uncensored-GGUF
Updated


In [16]:
#brochure_system_prompt = """
#You are an assistant that analyzes the contents of several relevant pages from a company website
#and creates a short brochure about the company for prospective customers, investors and recruits.
#Respond in markdown without code blocks.
#nclude details of company culture, customers and careers/jobs if you have the information.
#"""

#Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
 """


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b...
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nSearch models, datasets, users...\nCtrl K\nModels\nDatasets\nSpaces\nBuckets\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nthinkingmachines/Inkling-2-Mini\nUpdated\nabout 21 hours ago\n•\n41.9k\n•\n5.5

In [21]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="gpt-oss:20b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b...
Found 13 relevant links


# Hugging Face  
> **The AI community that’s *not* just hugging a few models.**  

---

## 🚀 **What We Do (and Why It Matters)**  
Hugging Face is the *go‑to* playground where *every* machine‑learning enthusiast, data scientist, and enterprise engineer get to **collaborate, experiment, and ship models faster than a coffee‑driven rabbit**.  

- **2 + million models** – From quirky chatbots to image‑to‑video maestros.  
- **500 k+ datasets** – Pixel art, cybersecurity, simulation assets, you name it.  
- **1 + million apps (Spaces)** – Run your models in the cloud without breaking a sweat.  
- **Open‑source library (🤗 Transformers)** – The backbone powering countless startups and research labs.

---

## 🤝 **Our Culture**  
- **Community‑first**: Think of us as a giant, friendly hack‑together‑in‑a‑coffee‑shop vibe.  
- **Open‑source at heart**: All core libraries and models are freely available; the only thing we’re “guarding” is our “Enterprise” tier, which gives you the extra horsepower you need.  
- **Zero bureaucracy, big ideas**: Pitch a new model in a Slack channel, get instant feedback, and ship it to production in 24 h (or less).  
- **Diversity + Inclusion**: A vibrant mix of developers from 70+ countries, languages, and walks of life.  

---

## 🌟 **Why Customers Love Us**  
| Who | What They Do | Hugging Face Helps |
|-----|--------------|--------------------|
| **Startups** | Build prototypes, prove concepts | 100 k+ free models, instant inference endpoints |
| **Enterprises** | Deploy AI at scale |  Enterprise support, inference providers, custom hosting |
| **Researchers** | Publish state‑of‑the‑art papers | Easy dataset versioning, model sharing, reproducibility |
| **Developers** | Add AI to apps | HuggingChat, Spaces, easy-to-use APIs |

---

## 🚀 **Spaces – Your Mini‑App Playground**  
- **Zero‑code or low‑code**: Drag, drop, or just write a tiny script.  
- **Instant deployment**: Click “Deploy” and your model runs on a GPU in the cloud.  
- **Community demos**: From generating poetry with GPT‑like models to turning 2D sketches into 3D objects.  

---

## 🛠️ **Careers – Join the Hugging Revolution**  
| Role | What’s Cool About It |
|------|----------------------|
| **Machine‑Learning Engineer** | Build next‑gen models that get a 1M+ star count. |
| **Open‑Source Lead** | Manage the world’s most popular ML libraries. |
| **Product & Evangelist** | Bring AI to everyday tools (and maybe convince the boss that “AI” is not a buzzword). |
| **Data Ops / DevOps** | Keep the inference engines humming while you sip coffee. |

> **Why we’re hiring** – Because we want *you* to write the next chapter of the AI revolution.  

---

## 📬 **Get in Touch**  
- **Website**: [huggingface.co](https://huggingface.co)  
- **Discord**: Hang out with 200k+ devs and researchers.  
- **GitHub**: Fork, contribute, and pull request.  
- **Enterprise**: Email us for custom solutions.  

---

## 🎉 **Final Thought**  
At Hugging Face, we’re building the *future* one model at a time, but we’re also building a community that’s as passionate about collaboration as it is about code. So, whether you’re looking to experiment, ship, or just have a good laugh at a “hug‑a‑model” joke, we’re here.  

> **Come join us – it’s not just a job, it’s a *hug* for the future!**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="gpt-oss:20b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b...
Found 14 relevant links


# Hugging Face – Where AI Gets a Big Friendly Hug

> **“The AI community building the future.”**  
> (And occasionally pulling the internet apart with a few lines of code.)

---

## 👋  Who Are We?

- **Founded**: 2016  
- **Employees**: 51‑200 (because we’re still a *small* but *powerful* squad)  
- **HQ**: Somewhere between “cloud” and “cafés with too‑much coffee.”  
- **Industry**: Software Development (specifically, *machine learning*).  

We’re a *software playground* where anyone from a curious data scientist to a bored intern can drop a model, dataset, or even a whole app and watch the community build a better world of AI—one Hug at a time.

---

## 🔬  What We Do

| **Feature** | **What It Looks Like** |
|-------------|------------------------|
| **Models** | 2 million+ models—yes, that’s *more* than there are people on Earth. |
| **Datasets** | 500 k+ datasets, from pixel‑art atlases to Uzbek language corpora. |
| **Spaces** | 1 M+ AI apps that run on **Zero Agents** or **Zero MCP** (because we love *zero* friction). |
| **Inference** | Endpoints, providers, and storage buckets that make cloud‑native deployment feel like a warm hug. |
| **Docs & Community** | Discord, GitHub, a blog, daily papers—just enough to keep the knowledge flowing. |

---

## 🤝  Our Culture

| **Trait** | **Why It Matters** |
|-----------|---------------------|
| **Open‑Source First** | We believe that the best models come from people who share them. |
| **Community‑Powered** | Every model, dataset, or Space gets a *Ctrl + K* spotlight and a friendly critique. |
| **Humor‑Heavy** | If your code fails, just say “I was hugging the wrong branch.” |
| **Fast‑Feedback Loop** | With the “Zero Agents” runtime, you can see your app run in seconds, not hours. |
| **Diversity & Inclusion** | From pixel‑art to quantum AI, we celebrate all kinds of creativity. |

---

## 🌍  Who Uses Us

- **Researchers** building the next transformer that can translate Uzbek to Icelandic (yes, that’s a thing).  
- **Startups** needing a quick inference endpoint to power their chatbot before the investor call.  
- **Gamers** creating realistic voice‑isolators for in‑game NPCs.  
- **Educators** who want a sandbox for teaching AI without buying a GPU farm.  

Our users love that they can **search, discover, and collaborate** on everything ML in one place—without the corporate buzzwords.

---

## 👩‍💻  Join the Hug

| **Role** | **Where You’ll Be** | **Why It’s Awesome** |
|----------|---------------------|----------------------|
| **Software Engineer** | Building next‑gen inference providers or polishing the UI for 2 M+ models. | Unlimited coffee + a bug‑free dev environment (almost). |
| **Data Curator** | Curating datasets like pixel‑art atlases or cybersecurity corpora. | Every dataset you publish becomes a living artifact. |
| **Community Lead** | Orchestrating Discord channels, writing docs, or moderating the forum. | You’ll shape how the world interacts with AI. |
| **Research Scientist** | Pushing the envelope on Mixture‑of‑Experts or new language models. | Your paper might land in the *Daily Papers* section. |

> *Open positions are listed on our Careers page. We’re always looking for people who can code, laugh, and occasionally hug their screen.*  

---

## 🎉  In a Nutshell

- **2 M+ models** = “We’re basically a library of every AI you can think of.”  
- **500 k+ datasets** = “We’ve got more data than your fridge has leftovers.”  
- **Zero Agents runtime** = “Run your Space in a fraction of a second—no lab coat required.”  
- **Open‑source community** = “Because sharing is caring, especially for transformers.”  

If you’re ready to join a company where you can *actually* help build the future of AI—and occasionally have a good laugh—drop us a line. We’ll hug you back with open-source code, endless coffee, and a seat at the table. 🚀

---

*Hugging Face – because the best way to build the future of AI is to share a hug (and a model).*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>